# Edges, Contours & Features Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Your first edges.** Canny = smooth -> gradient -> thinning -> hysteresis thresholds, all inside one call.

In [ ]:
import numpy as np
import cv2

part = np.full((200, 300), 90, dtype=np.uint8)
cv2.rectangle(part, (35, 45), (135, 155), 210, -1)
cv2.circle(part, (225, 100), 55, 205, -1)

smoothed = cv2.GaussianBlur(part, (5, 5), 0)     # stage 1 done here...
edges = cv2.Canny(smoothed, 60, 180)             # ...stages 2-4 inside Canny

print("edge pixels:", int((edges > 0).sum()))
# Canny = 1) smooth  2) gradient  3) thin to 1px ridges  4) hysteresis linking.

**2. Threshold tuning.** Low-low lets noise pose as edges, high-high kills real boundaries - aim for t2 ~ 2-3 x t1.

In [ ]:
import numpy as np
import cv2

part = np.full((200, 300), 90, dtype=np.uint8)
cv2.rectangle(part, (35, 45), (135, 155), 210, -1)
cv2.circle(part, (225, 100), 55, 205, -1)
rng = np.random.default_rng(0)
part = np.clip(part.astype(np.int16)
               + rng.normal(0, 8, part.shape).astype(np.int16),
               0, 255).astype(np.uint8)          # cheap sensor noise
smoothed = cv2.GaussianBlur(part, (5, 5), 0)

for t1, t2 in ((20, 40), (60, 180), (150, 400)):
    n = int((cv2.Canny(smoothed, t1, t2) > 0).sum())
    print(f"Canny({t1:3d}, {t2:3d}) -> {n:5d} edge pixels")
# (20, 40): faint noise clears the low bar and chains into hairlines -> clutter.
# (60, 180): the noise dies while real borders survive - the sweet spot.
# (150, 400): even genuine boundaries miss the high bar -> outlines vanish.

**3. Meet the contours.** Each contour is an `(N, 1, 2)` array of boundary points - OpenCV 4+ returns `(contours, hierarchy)`.

In [ ]:
import numpy as np
import cv2

part = np.full((200, 300), 90, dtype=np.uint8)
cv2.rectangle(part, (35, 45), (135, 155), 210, -1)
cv2.circle(part, (225, 100), 55, 205, -1)

_, bw = cv2.threshold(part, 127, 255, cv2.THRESH_BINARY)   # binary-ish input!

cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print("contours found:", len(cnts))
print("first contour:", cnts[0].shape, cnts[0].dtype)
print("first points :", cnts[0][:3].ravel())

## Part 2 — Practice

**4. Measure everything.** Contours come with free geometry: area, arcLength, moment centroid, boundingRect and a vertex-count signature.

In [ ]:
import numpy as np
import cv2

canvas = np.zeros((220, 320), dtype=np.uint8)
cv2.rectangle(canvas, (30, 50), (130, 170), 255, -1)
tri = np.array([(190, 40), (310, 100), (205, 190)], np.int32)
cv2.fillPoly(canvas, [tri], 255)

cnts, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

for c in sorted(cnts, key=cv2.contourArea, reverse=True):   # explicit order!
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)                           # True = closed
    M = cv2.moments(c)
    cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
    x, y, w, h = cv2.boundingRect(c)
    verts = len(cv2.approxPolyDP(c, 0.02 * peri, True))
    print(f"area={area:6.0f} peri={peri:6.1f} centroid=({cx},{cy}) "
          f"bbox=({x},{y}) {w}x{h} vertices={verts}")

**5. Holes need RETR_TREE.** EXTERNAL sees only outermost outlines; TREE adds inner boundaries and the parent links to navigate them.

In [ ]:
import numpy as np
import cv2

part = np.full((220, 340), 90, dtype=np.uint8)
cv2.rectangle(part, (35, 45), (140, 175), 210, -1)
cv2.rectangle(part, (70, 80), (105, 130), 90, -1)      # punch a hole
cv2.circle(part, (255, 110), 58, 205, -1)

_, bw = cv2.threshold(part, 127, 255, cv2.THRESH_BINARY)

ext, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
all_c, hier = cv2.findContours(bw, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

print("EXTERNAL contours:", len(ext))
print("TREE contours    :", len(all_c), "(outer outlines + hole)")

hole_idx = [i for i in range(len(all_c)) if hier[0][i][3] != -1]
print("hole contour index(es):", hole_idx)
# hierarchy rows are [Next, Previous, First_Child, Parent];
# a non-negative Parent means this curve lives INSIDE another shape.

**6. Sort and filter blobs.** List order follows scan position, not importance - sort by area and drop specks before any statistics.

In [ ]:
import numpy as np
import cv2

rng = np.random.default_rng(3)
scene = np.zeros((220, 360), dtype=np.uint8)
cv2.circle(scene, (60, 110), 15, 255, -1)       # small
cv2.circle(scene, (170, 80), 32, 255, -1)       # medium
cv2.circle(scene, (290, 140), 52, 255, -1)      # large
specks = rng.random(scene.shape) < 0.005        # tiny junk blobs
scene[specks] = 255

cnts, _ = cv2.findContours(scene, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
big = [c for c in sorted(cnts, key=cv2.contourArea, reverse=True)
       if cv2.contourArea(c) >= 500]

print("raw contours:", len(cnts), "-> after area filter:", len(big))
for c in big:
    x, y, w, h = cv2.boundingRect(c)
    print(f"  area={cv2.contourArea(c):7.0f}  bbox=({x},{y}) {w}x{h}")

## Part 3 — Challenge

**7. Build the shape classifier.** Vertex count IS the shape signature: polygons snap to their corner count, circles need many segments.

In [ ]:
import numpy as np
import cv2


def draw_shape(kind, size=160):
    img = np.zeros((size, size), dtype=np.uint8)
    cxy = size // 2
    if kind == "circle":
        cv2.circle(img, (cxy, cxy), size // 2 - 12, 255, -1)
    else:
        n = {"triangle": 3, "square": 4, "pentagon": 5}[kind]
        ang = np.linspace(0, 2 * np.pi, n, endpoint=False) - np.pi / 2
        if kind != "square":
            ang -= np.pi / n                 # sit flat like a real square
        r = size // 2 - 12
        pts = np.column_stack([cxy + r * np.cos(ang),
                               cxy + r * np.sin(ang)]).astype(np.int32)
        cv2.fillPoly(img, [pts], 255)
    return img


def classify(verts):                         # the whole "model"
    return {3: "triangle", 4: "square", 5: "pentagon"}.get(
        verts, "circle" if verts > 6 else f"{verts}-gon")


truth = ["triangle", "square", "pentagon", "circle"]
order = np.random.default_rng(42).permutation(truth)     # shuffle: no peeking

score = 0
for name in order:
    img = draw_shape(name)
    cnts, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)       # biggest blob wins
    verts = len(cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True))
    guess = classify(verts)
    score += guess == name
    print(f"true={name:9s} vertices={verts:2d} -> predicted: {guess}")

print(f"classifier score: {score}/{len(truth)}")

**8. Corners for tracking.** An edge looks the same when it slides along itself; a corner changes in every direction - that uniqueness is what trackers lock onto.

In [ ]:
import numpy as np
import cv2

board = np.kron([[1, 0] * 5, [0, 1] * 5] * 5,
                np.ones((40, 40))).astype(np.float32) * 255

harris = cv2.cornerHarris(board, blockSize=2, ksize=3, k=0.04)
harris_d = cv2.dilate(harris, None)               # fatten maxima for counting
hits = harris_d > 0.05 * harris_d.max()

corners = cv2.goodFeaturesToTrack(board, maxCorners=25,
                                  qualityLevel=0.05, minDistance=10)

print("Harris flagged pixels :", int(hits.sum()))
print("trackable points kept :", len(corners))
# Edges are ambiguous along their own direction; corners are unique in BOTH,
# so optical flow can match them frame after frame.

**9. Find the sprite (template matching).** Slide-and-score finds exact-size copies perfectly - but offers zero scale or rotation invariance.

In [ ]:
import numpy as np
import cv2

rng = np.random.default_rng(11)
scene_f = rng.random((260, 380)) * 60              # busy low-gray clutter
scene = cv2.GaussianBlur(scene_f.astype(np.uint8), (3, 3), 0)

sprite = np.zeros((36, 48), dtype=np.uint8)
pts = np.array([[4, 18], [28, 4], [28, 12], [44, 12],
                [44, 24], [28, 24], [28, 32]], np.int32)
cv2.fillPoly(sprite, [pts], 255)

positions = [(60, 40), (230, 150)]
for x, y in positions:
    scene[y:y + 36, x:x + 48] = sprite

res = cv2.matchTemplate(scene, sprite, cv2.TM_CCOEFF_NORMED)
mn, mx, mnloc, mxloc = cv2.minMaxLoc(res)
print(f"best score {mx:.3f} at (x={mxloc[0]}, y={mxloc[1]})")

h, w = sprite.shape
ys, xs = np.where(res >= 0.85)                     # every strong hit
boxes = []
for x, y in zip(xs, ys):
    if all(abs(x - bx) > w // 2 or abs(y - by) > h // 2 for bx, by in boxes):
        x0, y0 = max(0, x - w // 2), max(0, y - h // 2)
        win = res[y0:y0 + h, x0:x0 + w]            # one peak region...
        dy, dx = np.unravel_index(win.argmax(), win.shape)
        boxes.append((int(x0 + dx), int(y0 + dy))) # ...snapped to its maximum

print("detected:", sorted(boxes))
print("true    :", sorted(positions))
# Works because the sprite appears at IDENTICAL size and angle;
# resize or rotate the target and the score collapses silently.